# Step 4 - Evaluation and Interpretability

This notebook reloads the Step 3 checkpoints (no retraining)
and produces the evidence for the report: scores per seed and per class, how
errors relate to graph homophily, what the gate learned, and which genes the
models rely on. Run it after notebooks 01 -> 02 -> 03. The functions live in
`src/interpretability.py`; every table and figure is also saved to
`data/interpretability/`.

**Scope.** By default, every section covers the 108 checkpoints produced by
submission notebook 03: 4 settings in `main`, 6 in `scarcity` and 4 in `depth`.
A setting is one (experiment, budget, graph, depth) combination. Set
`RUN_NEIGHBOURHOOD = True` only when the additional graphs and 48 neighbourhood
checkpoints are available. Sections 2-7 report each setting and every model in
it, averaged over seeds. Gate analyses apply only to the gated GCN.
Per-setting figures are saved in subfolders named after the section.

**MLP baseline.** Read notebook 02's existing three-seed validation scores at
5pct and full from `data/mlp_50pc/`; this notebook does not retrain the MLP.
Notebook 02 exports per-cell predictions only for seed 42, so MLP confusion
matrices and dropout comparisons use that seed. GNN results average three seeds.
There is no MLP baseline at the other budgets.

**Names.** Figures use the report's names: MLP, GCN, GraphSAGE, GAT, Gated GCN.

**Workflow**

| Step | Reads | Functions | Produces |
|---|---|---|---|
| Setup | Step 2 exports: PCs, masks, graphs | `DataCache` | `cache`: one dataset per budget and graph |
| 1. Scores | Step 3 checkpoints (`CHECKPOINT_DIRS`), notebook 02 MLP scores and predictions | `run_grid`, `evaluate_runs`, `group_runs`, `load_mlp_baseline` | `results` (one row per run), `outputs` (per-cell predictions and gate per run), `grouped` (setting -> model -> seed), `mlp_runs` (MLP predictions per budget and seed), `mlp` |
| 1. Final model | `FINAL_CHECKPOINT` | `RunSpec.from_checkpoint`, `evaluate_runs` | `final` (test scores) |
| 2. Confusion | `grouped` | `mean_confusion`, `plot_confusions`, `granularity_table` | `confusion/`, `granularity` |
| 3. Homophily | `results`, both training graphs | `local_homophily`, `class_homophily`, `per_class_vs_homophily`, `homophily_tests` | `local` (per graph and cell), `per_class_all`, `tests` |
| 4. Gate | gated GCN runs in `grouped`, `local` | `gate_vs_homophily` | `gate_summary`, `within_class` |
| 5. Attribution | checkpoints, `work_processed.h5ad`, PCs | `load_expression`, `mean_attribution`, `marker_recovery` | `recovery` |
| 6. UMAP | PCs, `grouped`, `local` | `umap_coords`, `plot_umap` | `umap/` |
| 7. Dropout | `grouped`, `mlp_runs`, `work_processed.h5ad` | `genes_detected`, `dropout_table` | `dropout` |
| 8. Report figures | results of sections 1-7 | `report_*` | `report/`: one graph per image |

**Splits.** Sections 1-7 compare and interpret models on the validation split.
The last cell of section 1 scores the final checkpoint once on the test split;
those are the headline numbers for the report.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import interpretability as interp

# Inputs
PROCESSED_H5AD = ROOT / "data/processed/work_processed.h5ad"  # notebook 01
EXPORT_ROOT = ROOT / "data/graph_engineering_50pc"  # notebook 02: PCs, masks, graphs
GNN_ROOT = ROOT / "data/gnn_training_50pc"  # notebook 03
CHECKPOINT_DIRS = {
    "main": GNN_ROOT,
    "scarcity": GNN_ROOT / "scarcity",
    "depth": GNN_ROOT / "depth",
    "neighbourhood": GNN_ROOT / "neighbourhood",
}
# Final model selected by validation macro-F1 in the local 50-PC run
FINAL_CHECKPOINT = GNN_ROOT / "scarcity/gated_gcn_20pct_knn_depth2_seed2024.pt"

MLP_ROOT = ROOT / "data/mlp_50pc"  # notebook 02: three-seed validation scores

# Outputs
OUT = ROOT / "data/interpretability"
OUT.mkdir(parents=True, exist_ok=True)

# The Step 3 grid
SEEDS = [42, 123, 2024]
MODELS = ["gcn", "graphsage", "gat", "gated_gcn"]
ABLATION_MODELS = ["gcn", "gated_gcn"]
BUDGETS = ["1pct", "2pct", "5pct", "10pct", "20pct", "full"]
DEPTHS = [1, 2, 3, 4]
# Enable only after generating the additional graphs and checkpoints.
RUN_NEIGHBOURHOOD = False
NEIGHBOUR_GRAPHS = ["knn5", "knn10", "knn15", "knn30"]
UMAP_CELLS = 15000

cache = interp.DataCache(EXPORT_ROOT)  # loads each (budget, graph) export once


def save_and_show(fig, path):
    """Save a figure, show it inline and free its memory."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    display(fig)
    plt.close(fig)

## 1. Scores

In [ ]:
specs = (
    interp.run_grid("main", MODELS, ["5pct", "full"], ["knn", "snn"], [2], SEEDS)
    + interp.run_grid("scarcity", ABLATION_MODELS, BUDGETS, ["knn"], [2], SEEDS)
    + interp.run_grid("depth", ABLATION_MODELS, ["5pct"], ["knn"], DEPTHS, SEEDS)
)
if RUN_NEIGHBOURHOOD:
    specs += interp.run_grid(
        "neighbourhood", ABLATION_MODELS, ["5pct", "full"], NEIGHBOUR_GRAPHS, [2], SEEDS
    )
results, outputs = interp.evaluate_runs(specs, cache, CHECKPOINT_DIRS, "validation")
results.to_csv(OUT / "validation_scores_by_run.csv", index=False)

grouped = interp.group_runs(outputs)
SETTINGS = list(grouped)
print(f"{len(outputs)} runs in {len(SETTINGS)} settings")

In [ ]:
# Reuse notebook 02's trained MLP; no fitting takes place in this notebook.
mlp = interp.load_mlp_baseline(MLP_ROOT)
mlp_runs = {
    budget: {42: np.load(EXPORT_ROOT / f"results/{budget}__mlp_predictions.npy")}
    for budget in ["5pct", "full"]
}
# Notebook 02 exports all-cell predictions only for seed 42. Confusion and
# dropout therefore compare this MLP seed against the GNN mean over three seeds.
mlp.to_csv(OUT / "validation_mlp_scores_by_run.csv", index=False)
summary = interp.seed_summary(results)
summary.to_csv(OUT / "validation_summary.csv", index=False)
print("Validation macro-F1, mean ± sd over seeds (the MLP uses the same seeds):")
display(interp.headline_table(summary, mlp, MODELS, BUDGETS))

In [ ]:
paired = pd.concat(
    interp.paired_seed_delta(results, "gated_gcn", other).assign(
        comparison=f"gated_gcn - {other}"
    )
    for other in MODELS
    if other != "gated_gcn"
)
paired.to_csv(OUT / "validation_paired_deltas.csv", index=False)
print("Per-seed macro-F1 differences on identical settings:")
display(paired.round(3))

In [ ]:
fig = interp.plot_overview(results, mlp, BUDGETS)
save_and_show(fig, OUT / "validation_overview.png")

In [ ]:
final, _ = interp.evaluate_runs(
    [interp.RunSpec.from_checkpoint(FINAL_CHECKPOINT)], cache, CHECKPOINT_DIRS, "test"
)
final.to_csv(OUT / "test_final_model.csv", index=False)
metrics = ["accuracy", "macro_f1", *[c for c in final if c.startswith("f1::")]]
display(final[metrics].T.set_axis(["test"], axis=1).round(3))

## 2. Confusion structure and label granularity

In [ ]:
base = cache.get("full", "knn")
class_names = list(base.class_names)
labels = interp.all_labels(base)
cell_types = np.asarray(base.cell_types)
for setting in SETTINGS:
    data = cache.get(setting.budget, setting.graph)
    assert list(data.class_names) == class_names
    assert np.array_equal(data.cell_ids, base.cell_ids)

display(
    pd.DataFrame(
        [
            {**setting.columns, "models": ", ".join(runs), "seeds": len(SEEDS)}
            for setting, runs in grouped.items()
        ]
    )
)

In [ ]:
for setting, runs in grouped.items():
    tables = {
        model: interp.mean_confusion(seeds, class_names)
        for model, seeds in runs.items()
    }
    fig = interp.plot_confusions(tables, f"{setting.name}, mean of {len(SEEDS)} seeds")
    save_and_show(fig, OUT / "confusion" / f"{setting.name}.png")

In [ ]:
granularity = interp.granularity_table(outputs, cache)
granularity.to_csv(OUT / "validation_granularity.csv")
display(granularity.round(3))

## 3. Per-class F1 vs homophily

In [ ]:
GRAPHS = list(dict.fromkeys(setting.graph for setting in SETTINGS))
adjacency = {
    graph: sp.load_npz(EXPORT_ROOT / "graphs" / f"{graph}_adjacency.npz")
    for graph in GRAPHS
}
local = {  # per graph and cell; reused in 4 and 6
    graph: interp.local_homophily(matrix, labels) for graph, matrix in adjacency.items()
}
homophily = {
    graph: interp.class_homophily(matrix, labels, class_names)
    for graph, matrix in adjacency.items()
}

per_class_tables, test_tables = {}, []
for setting, runs in grouped.items():
    per_class = interp.per_class_vs_homophily(
        results, homophily[setting.graph], class_names, setting, mlp
    )
    per_class_tables[setting] = per_class
    quantities = [*runs, *(["mlp"] if "mlp" in per_class else []), "gate_gain"]
    test_tables.append(
        interp.homophily_tests(per_class, quantities).assign(**setting.columns)
    )

per_class_all = pd.concat(
    {setting.name: table for setting, table in per_class_tables.items()},
    names=["setting", "celltype"],
)
per_class_all.to_csv(OUT / "validation_per_class_vs_homophily.csv")
tests = pd.concat(test_tables, ignore_index=True)

print("F1 gain of the gated GCN over the GCN per class and setting:")
display(
    per_class_all["gate_gain"]
    .unstack("setting")[[setting.name for setting in SETTINGS]]
    .reindex(class_names)
    .round(3)
)
print("Spearman rho against class endpoint homophily (permutation p-values):")
display(
    tests.pivot_table(
        index=list(interp.SETTING_KEYS),
        columns="quantity",
        values=["rho", "p_permutation"],
        sort=False,
    ).round(3)
)
if RUN_NEIGHBOURHOOD:
    size = pd.read_csv(EXPORT_ROOT / "analysis/neighbourhood_size.csv")
    print("Graph size and edge homophily by neighbourhood size k:")
    display(
        size.drop_duplicates("k")
        .set_index("k")[["edges", "mean_degree", "components", "global_edge_homophily"]]
        .round(3)
    )
    display(
        size.pivot(index="celltype", columns="k", values="endpoint_homophily")
        .reindex(class_names)
        .round(3)
    )

In [ ]:
for setting, per_class in per_class_tables.items():
    fig = interp.plot_per_class_vs_homophily(
        per_class, list(grouped[setting]), setting.name
    )
    save_and_show(fig, OUT / "per_class_vs_homophily" / f"{setting.name}.png")

## 4. Gate vs homophily

In [ ]:
gate_rows, within_tables, class_mean_tables = [], {}, {}
for setting, runs in grouped.items():
    rows, within, means = interp.gate_vs_homophily(
        runs["gated_gcn"], local[setting.graph], cell_types, class_names
    )
    gate_rows.append(rows.assign(**setting.columns))
    within_tables[setting.name] = within
    class_mean_tables[setting] = means

gate_summary = pd.concat(gate_rows, ignore_index=True)
within_class = pd.concat(within_tables, names=["setting"])
gate_summary.to_csv(OUT / "validation_gate_vs_homophily.csv", index=False)
within_class.to_csv(OUT / "validation_gate_within_class.csv")

print("Gate vs local homophily, mean over seeds:")
display(
    gate_summary.groupby(list(interp.SETTING_KEYS), sort=False)[
        ["overall_rho", "class_level_rho", "class_level_p", "median_within_class_rho"]
    ]
    .mean()
    .round(4)
)
print("Within-class rho per setting, mean over seeds:")
display(
    within_class["rho"]
    .unstack("setting")[list(within_tables)]
    .reindex(class_names)
    .round(3)
)

In [ ]:
for setting, means in class_mean_tables.items():
    run = grouped[setting]["gated_gcn"][SEEDS[0]]
    fig = interp.plot_gate_vs_homophily(
        run, local[setting.graph], means, SEEDS[0], setting.name
    )
    save_and_show(fig, OUT / "gate_vs_homophily" / f"{setting.name}.png")

## 5. Gradient attribution through PCA loadings

In [ ]:
pcs = np.load(EXPORT_ROOT / "X_pca.npy", allow_pickle=False)  # unscaled model input
loadings, gene_names, hvg, expression = interp.load_expression(
    PROCESSED_H5AD, base.cell_ids
)
print(f"Genes: {len(gene_names):,}; HVGs: {hvg.sum():,}; loadings: {loadings.shape}")

# Attributions are only valid if loadings, expression and PCs belong together.
min_corr = interp.check_pca_loadings(expression, loadings, hvg, pcs)
print(f"Min PC correlation: {min_corr:.4f}")
assert min_corr > 0.99, "Loadings do not reproduce the PCs."

In [ ]:
# load_data standardises the PCs with a StandardScaler fitted on labelled cells.
eligible = pd.Series(hvg, index=gene_names)
recovery_tables = []
for setting, runs in grouped.items():
    data = cache.get(setting.budget, setting.graph)
    scale = pcs[data.train_mask].astype(np.float64).std(axis=0)
    groups = {
        model: [
            interp.load_model(run.spec, data, CHECKPOINT_DIRS) for run in seeds.values()
        ]
        for model, seeds in runs.items()
    }
    groups["untrained_gated_gcn"] = [
        interp.untrained_model("gated_gcn", data, setting.depth)
    ]
    for name, group in groups.items():
        table = interp.mean_attribution(
            group, data, data.validation_mask, scale, loadings, gene_names
        )
        recovery_tables.append(
            interp.marker_recovery(table, eligible).assign(
                **setting.columns, model=name
            )
        )

recovery = pd.concat(recovery_tables, ignore_index=True)
recovery.to_csv(OUT / "validation_marker_recovery.csv", index=False)
print("Markers per class that reach the model input:")
display(
    recovery.drop_duplicates(["marker", "celltype"])
    .pivot_table(index="celltype", columns="in_hvg", values="marker", aggfunc=", ".join)
    .rename(columns={True: "HVG (scored)", False: "not HVG or removed in QC"})
    .reindex(class_names)
)
print("Marker rank as top percent of HVGs (lower is better):")
display(
    recovery[recovery["in_hvg"]]
    .pivot_table(
        index=[*interp.SETTING_KEYS, "model"],
        columns=["marker", "celltype"],
        values="top_percent",
        sort=False,
    )
    .round(1)
)

## 6. UMAPs

In [ ]:
subset, coords = interp.umap_coords(
    labels, pcs, OUT / f"umap_{UMAP_CELLS}.npz", UMAP_CELLS
)
truth = cell_types[subset]
names = np.asarray(class_names)
for setting, runs in grouped.items():
    colourings = {
        "celltype": truth,
        f"local_homophily ({setting.graph})": local[setting.graph][subset],
    }
    for model, seeds in runs.items():
        colourings[f"{interp.model_label(model)}: share of seeds correct"] = np.mean(
            [names[run.predictions[subset]] == truth for run in seeds.values()],
            axis=0,
        )
    colourings["Gated GCN: gate, mean over seeds"] = np.mean(
        [run.gate[subset] for run in runs["gated_gcn"].values()], axis=0
    )
    fig = interp.plot_umap(
        coords, colourings, f"UMAP; {setting.name}, {len(SEEDS)} seeds"
    )
    save_and_show(fig, OUT / "umap" / f"{setting.name}.png")

## 7. Dropout

In [ ]:
# If neighbour averaging smooths dropout, gnn_minus_mlp is highest in bin 1.
n_genes = interp.genes_detected(PROCESSED_H5AD, base.cell_ids)
dropout_by_setting = {
    setting: interp.dropout_table(
        runs, mlp_runs.get(setting.budget), n_genes, cell_types
    )  # The exported MLP baseline uses seed 42
    for setting, runs in grouped.items()
}
dropout = pd.concat(
    [
        table.reset_index().assign(**setting.columns)
        for setting, table in dropout_by_setting.items()
    ],
    ignore_index=True,
)
dropout.to_csv(OUT / "validation_dropout.csv", index=False)

for value, title in [
    ("accuracy_mean", "GNN validation accuracy per dropout bin (1 = most dropout):"),
    ("gnn_minus_mlp_mean", "GNN minus MLP accuracy per dropout bin:"),
]:
    print(title)
    display(
        dropout.pivot_table(
            index=[*interp.SETTING_KEYS, "model"],
            columns="bin",
            values=value,
            sort=False,
        ).round(3)
    )

## 8. Report figures

One graph per image, sized for one column of the report, using the results of
sections 1-7. They cover the main $k$-NN settings at depth 2 (5pct and full)
and the scarcity, neighbourhood and depth ablations. Saved to `report/`.

In [ ]:
REPORT = OUT / "report"
main = {
    setting.budget: setting
    for setting in SETTINGS
    if setting.experiment == "main" and setting.graph == "knn"
}

# Per-class F1 against homophily, and the gain of the gate per class
for budget, setting in main.items():
    title = f"Per-class F1, {interp.budget_label(budget)} labels, $k$-NN"
    fig = interp.report_class_f1_vs_homophily(per_class_tables[setting], MODELS, title)
    save_and_show(fig, REPORT / f"class_f1_vs_homophily_{budget}_knn.png")
fig = interp.report_gate_gain_by_class(
    {budget: per_class_tables[setting] for budget, setting in main.items()},
    "Per-class gain of Gated GCN over GCN ($k$-NN)",
)
save_and_show(fig, REPORT / "gate_gain_by_class_knn.png")

# The gate
fig = interp.report_mean_gate_by_budget(results, BUDGETS)
save_and_show(fig, REPORT / "mean_gate_by_budget_knn.png")
run = grouped[main["full"]]["gated_gcn"][SEEDS[0]]
fig = interp.report_gate_cells(
    run, local["knn"], f"Validation cells, Full labels, seed {SEEDS[0]}"
)
save_and_show(fig, REPORT / f"gate_cells_full_knn_seed{SEEDS[0]}.png")
for budget, setting in main.items():
    rows = gate_summary[
        (gate_summary["experiment"] == "main")
        & (gate_summary["budget"] == budget)
        & (gate_summary["graph"] == "knn")
    ]
    fig = interp.report_gate_class_means(
        class_mean_tables[setting],
        rows["class_level_rho"].mean(),
        rows["class_level_p"].mean(),
        f"Class means, {interp.budget_label(budget)} labels, $k$-NN",
    )
    save_and_show(fig, REPORT / f"gate_class_means_{budget}_knn.png")

# Confusion matrices: MLP seed 42 and GNN means over three seeds
data_5 = cache.get("5pct", "knn")
mask, targets = data_5.validation_mask, interp.all_labels(data_5)
confusions = {
    "mlp": sum(
        interp.confusion_table(pred[mask], targets[mask], class_names)
        for pred in mlp_runs["5pct"].values()
    )
    / len(mlp_runs["5pct"])
}
for model in MODELS:
    confusions[model] = interp.mean_confusion(grouped[main["5pct"]][model], class_names)
for model, table in confusions.items():
    seed_label = "seed 42" if model == "mlp" else "mean over three seeds"
    title = f"{interp.model_label(model)}, 5% labels, $k$-NN, {seed_label}"
    fig = interp.report_confusion(table, title)
    save_and_show(fig, REPORT / f"confusion_5pct_knn_{model}.png")

# Ablations and dropout
save_and_show(interp.report_scarcity(results, mlp, BUDGETS), REPORT / "scarcity_knn.png")
if RUN_NEIGHBOURHOOD:
    save_and_show(interp.report_neighbourhood(results), REPORT / "neighbourhood_size_knn.png")
save_and_show(interp.report_depth(results), REPORT / "depth_5pct_knn.png")
for budget, setting in main.items():
    title = f"Dropout, {interp.budget_label(budget)} labels, $k$-NN (MLP seed 42)"
    fig = interp.report_dropout_gap(dropout_by_setting[setting], title)
    save_and_show(fig, REPORT / f"dropout_gap_{budget}_knn.png")
print(sorted(path.name for path in REPORT.glob("*.png")))

## Interpretation notes

TODO: Include the following points in the report, with references to the notebook sections and figures.

1. **Headline.** What are the final model's test macro-F1 and per-class F1
   (end of section 1)? On validation, does any GNN beat the MLP (section 1), by more
   than the seed spread?
2. **Gate effect.** Is gated minus GCN positive for every seed and every setting?
   Does it shrink as labels increase (scarcity rows) or as depth grows (depth rows)?
3. **Gate vs GraphSAGE and GAT.** If GraphSAGE matches or beats the gated GCN, the
   gain comes from keeping self features, not from adaptive gating. GAT shows
   whether learned edge attention does the same job as the gate.
4. **Granularity.** How much of the error is within the T-cell family and does
   that share change across budgets and graphs?
5. **Homophily link.** Do low-homophily classes have low F1, and does the gate
   help them most? Does the pattern hold on both the kNN and the SNN graph?
6. **Gate mechanism.** Is the within-class rho negative? If only the pooled rho
   is, the gate tracks cell type, not neighbourhood reliability. Check that the
   sign is the same in every setting.
7. **Biology.** Do trained models rank the canonical markers higher than the
   untrained gated GCN? Most T-cell markers (CD3E, CD8A, CCR7, FOXP3 and others)
   are not HVGs so no model can use them. CD34 was removed in QC. Which classes
   still have a scored marker?
8. **Dropout.** Is gnn_minus_mlp highest in bin 1 (fewest genes
   detected)? If it is flat across bins, neighbour averaging does not act
   as a dropout smoother. Only 5pct and full have an MLP baseline.
9. **Consistency.** A finding that holds in only one setting should be reported
   as such. The 5pct kNN depth-2 setting appears in all three experiments.
   It also shows how much separately trained runs of the same setting differ.
10. **Neighbourhood size.** Does a larger k lower homophily and hurt the GCN
    more than the gated GCN? Does a small k cut off rare classes?
11. **Limitations and future work.** Transductive single-donor setting, three
    seeds, validation-selected checkpoints, label noise in the Zheng 68k
    annotations.